# SimWorld Studio

**AI-powered 3D scene generation platform** — chat with DeepSeek, Qwen, or another OpenAI-compatible model to build urban scenes in real time.

## What this notebook does
1. Checks your Colab GPU
2. Downloads the minimal SimWorld binary (~15 GB)
3. Installs SimWorld Studio platform
4. Sets up LLM authentication (DeepSeek or Qwen/DashScope API key)
5. Launches everything and gives you a browser URL

**Run all cells in order.** Total setup time: ~5 minutes.

---

## Cell 1: GPU Check

In [ ]:
"""Verify GPU is available — T4 or better required."""
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
if result.returncode != 0:
    print("ERROR: No GPU detected!")
    print("Go to: Runtime -> Change runtime type -> GPU (T4)")
    sys.exit(1)

gpu_info = result.stdout.strip()
print(f"GPU: {gpu_info}")

supported = ["T4", "A100", "V100", "L4", "A10"]
if not any(g in gpu_info for g in supported):
    print(f"WARNING: Unrecognized GPU. Supported: {supported}")
    print("Continuing anyway...")
else:
    print("GPU check passed!")

## Cell 2: Download SimWorld (Minimal Binary)

Downloads a minimal UE Editor binary (~15 GB compressed) with just the assets needed for the Studio demo.

In [ ]:
"""Download and extract the minimal SimWorld binary."""
import os

SIMWORLD_DIR = "/content/SimWorld-Studio-Minimal"
SIMWORLD_ARCHIVE = "/tmp/SimWorld-Studio-Minimal.tar.gz"
DOWNLOAD_URL = "https://huggingface.co/datasets/SimWorld-AI/SimWorld-Studio/resolve/main/SimWorld-Studio-Minimal.tar.gz"

UE_BINARY = os.path.join(SIMWORLD_DIR, "Engine", "Binaries", "Linux", "UnrealEditor")

if os.path.exists(UE_BINARY):
    print(f"SimWorld binary already installed at: {SIMWORLD_DIR}")
else:
    print("Downloading SimWorld minimal binary (~15 GB)...")
    !wget -q --show-progress -O {SIMWORLD_ARCHIVE} {DOWNLOAD_URL}
    print("Extracting...")
    !tar xzf {SIMWORLD_ARCHIVE} -C /content/
    !rm -f {SIMWORLD_ARCHIVE}

    if os.path.exists(UE_BINARY):
        os.chmod(UE_BINARY, 0o755)
        # Also chmod the launch script
        launch_sh = os.path.join(SIMWORLD_DIR, "SimWorld-Studio.sh")
        if os.path.exists(launch_sh):
            os.chmod(launch_sh, 0o755)
        print(f"SimWorld binary installed at: {SIMWORLD_DIR}")
    else:
        print("ERROR: UnrealEditor not found after extraction!")
        !find {SIMWORLD_DIR} -maxdepth 3 -type f -name "UnrealEditor*" | head -5

print(f"\nUE Binary: {UE_BINARY}")
print(f"Project: {os.path.join(SIMWORLD_DIR, 'gym_citynav', 'gym_citynav.uproject')}")

## Cell 3: Install Studio Platform

In [ ]:
"""Install the Studio platform."""
import subprocess, shutil, os

# --- Studio platform (pip package from GitHub) ---
print("[1/2] Installing SimWorld Studio...")

STUDIO_GIT_URL = "git+https://github.com/dddaaamax/SimWorld-Studio-main.git#subdirectory=packaging"

result = subprocess.run(
    ["pip", "install", "-q", STUDIO_GIT_URL],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("  Installed from GitHub.")
else:
    print(f"  ERROR: Could not install package.")
    print(f"  {result.stderr[:200]}")
    print("  Please check the GitHub URL or install manually.")

# Verify installation
studio_installed = shutil.which("simworld-studio") is not None
if studio_installed:
    ver = subprocess.run(["simworld-studio", "version"], capture_output=True, text=True).stdout.strip()
    print(f"  {ver}")
else:
    print("  WARNING: simworld-studio CLI not found after install!")

# --- Node.js (usually pre-installed in Colab) ---
print("[2/2] Checking Node.js...")
if not shutil.which("node"):
    print("  Installing Node.js...")
    !apt-get install -y -qq nodejs npm
else:
    node_ver = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"  Node.js {node_ver} found.")

print("\nAll dependencies installed!")

## Cell 4: Choose LLM Provider

Default is **DeepSeek**. You can switch to **Qwen/DashScope** by changing `provider` to `qwen`.

Your API key stays in this Colab runtime environment and is passed only to the selected model provider.

In [ ]:
"""Configure DeepSeek or Qwen/DashScope for SimWorld Studio."""
import os
from getpass import getpass

# Choose provider: "deepseek" or "qwen".
provider = "deepseek"

if provider == "deepseek":
    os.environ["SIMWORLD_LLM_PROVIDER"] = "deepseek"
    os.environ["SIMWORLD_LLM_MODEL"] = "deepseek-chat"  # or "deepseek-reasoner"
    if not os.environ.get("DEEPSEEK_API_KEY"):
        os.environ["DEEPSEEK_API_KEY"] = getpass("Enter your DeepSeek API key: ").strip()
    print(f"DeepSeek configured: model={os.environ['SIMWORLD_LLM_MODEL']}")
elif provider == "qwen":
    os.environ["SIMWORLD_LLM_PROVIDER"] = "qwen"
    os.environ["SIMWORLD_LLM_MODEL"] = "qwen-plus"  # examples: qwen-plus, qwen-max, qwen-turbo
    if not os.environ.get("DASHSCOPE_API_KEY"):
        os.environ["DASHSCOPE_API_KEY"] = getpass("Enter your DashScope API key: ").strip()
    print(f"Qwen/DashScope configured: model={os.environ['SIMWORLD_LLM_MODEL']}")
else:
    raise ValueError("provider must be 'deepseek' or 'qwen'")

print("LLM credentials are stored only in this Colab runtime's environment.")

## Cell 5: Launch SimWorld + Studio

In [ ]:
"""Start SimWorld (headless GPU) and the Studio backend."""
import subprocess, time, os, requests

# --- Install headless rendering dependencies ---
print("[1/3] Installing rendering dependencies...")
r = subprocess.run(
    ["apt-get", "install", "-y", "-qq",
     "xvfb", "vulkan-tools", "mesa-vulkan-drivers",
     "libvulkan1", "libegl1", "libgles2", "libgl1"],
    capture_output=True
)
if r.returncode != 0:
    print(f"  WARNING: Some rendering deps may have failed (exit {r.returncode})")
else:
    print("  Rendering deps installed.")

# --- Start virtual display ---
print("[2/3] Starting virtual display...")
subprocess.run(["pkill", "-f", "Xvfb"], capture_output=True)
time.sleep(1)
xvfb_proc = subprocess.Popen(
    ["Xvfb", ":99", "-screen", "0", "1280x720x24"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
os.environ["DISPLAY"] = ":99"
time.sleep(2)
print("  Virtual display :99 started.")

# --- Launch SimWorld + Studio platform ---
print("[3/3] Launching SimWorld Studio (this can take 1-2 minutes)...")
SIMWORLD_DIR = "/content/SimWorld-Studio-Minimal"
MCP_PORT = 55559

studio_proc = subprocess.Popen(
    ["simworld-studio", "start",
     "--binary", SIMWORLD_DIR,
     "--mcp-port", str(MCP_PORT),
     "--port", "3002",
     "--map", "/Game/Maps/Empty",
     "--data-dir", "/content/studio_workspace"],
    stdout=open("/content/studio.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

# Verify backend is up
for attempt in range(90):
    try:
        r = requests.get("http://localhost:3002/api/health", timeout=5)
        print(f"  Studio backend running! Health: {r.json()}")
        break
    except Exception:
        if attempt % 10 == 0:
            print(f"  Still starting... ({attempt * 2}s)")
        time.sleep(2)
else:
    print("  Studio backend did not answer yet. Check: !tail -80 /content/studio.log")

print("\nAll services launched!")

## Cell 6: Get Your Browser URL

In [ ]:
"""Create a public tunnel URL to access the Studio in your browser."""
import subprocess, time, re, os, select

# Install cloudflared
print("Setting up tunnel...")
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        ["wget", "-q",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", "/usr/local/bin/cloudflared"],
        check=True
    )
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

# Kill any existing tunnel
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# Start tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:3002", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for URL — use select() for non-blocking reads
public_url = None
accumulated = ""
for _ in range(30):  # wait up to 30 seconds
    time.sleep(1)
    try:
        ready, _, _ = select.select([tunnel.stderr], [], [], 0)
        if ready:
            chunk = os.read(tunnel.stderr.fileno(), 8192).decode(errors='replace')
            accumulated += chunk
            match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', accumulated)
            if match:
                public_url = match.group(0)
                break
    except Exception:
        pass

if public_url:
    print(f"")
    print(f"{'='*60}")
    print(f"  SimWorld Studio is live!")
    print(f"")
    print(f"  Open in browser: {public_url}")
    print(f"{'='*60}")
    print(f"")
    print(f"Chat with DeepSeek/Qwen to build 3D urban scenes!")
    print(f"Try: 'Build a small neighborhood with 4 houses and trees'")
else:
    print("Tunnel failed to start. Trying Colab proxy fallback...")
    try:
        from google.colab.output import eval_js
        proxy_url = eval_js("google.colab.kernel.proxyPort(3002)")
        public_url = proxy_url
        print(f"Open in browser: {proxy_url}")
    except Exception:
        print("Both tunnel methods failed. Check your connection.")
        public_url = "http://localhost:3002"

## Cell 7: Verify Everything Works

Run this cell to confirm all services are healthy before using the Studio.

In [ ]:
"""Automated verification — all checks must pass before using the Studio."""
import requests, socket, subprocess, os

print("Running verification checks...\n")

results = {}

# 1. GPU
r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                   capture_output=True, text=True)
results["GPU"] = (r.returncode == 0, r.stdout.strip() if r.returncode == 0 else "Not found")

# 2. SimWorld binary
ue_bin = "/content/SimWorld-Studio-Minimal/Engine/Binaries/Linux/UnrealEditor"
results["SimWorld Binary"] = (os.path.exists(ue_bin), ue_bin if os.path.exists(ue_bin) else "Not found")

# 3. SimWorld TCP (MCP port)
mcp_port = globals().get('MCP_PORT', 55559)
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(5)
    s.connect(("127.0.0.1", mcp_port))
    s.close()
    results[f"SimWorld MCP ({mcp_port})"] = (True, "Connected")
except Exception as e:
    results[f"SimWorld MCP ({mcp_port})"] = (False, str(e))

# 4. Studio Backend
try:
    r = requests.get("http://localhost:3002/api/health", timeout=10)
    results["Studio Backend"] = (r.status_code == 200, f"HTTP {r.status_code}")
except Exception as e:
    results["Studio Backend"] = (False, str(e))

# 5. Skills
try:
    r = requests.get("http://localhost:3002/api/skills", timeout=10)
    data = r.json()
    count = len(data) if isinstance(data, list) else 0
    results["Skills"] = (count > 0, f"{count} skills loaded")
except Exception as e:
    results["Skills"] = (False, str(e))

# 6. Assets
try:
    r = requests.get("http://localhost:3002/api/assets", timeout=10)
    results["Assets"] = (r.status_code == 200, "Catalog loaded")
except Exception as e:
    results["Assets"] = (False, str(e))

# 7. LLM provider
provider = (os.environ.get("SIMWORLD_LLM_PROVIDER") or "").lower()
if not provider:
    provider = "deepseek" if os.environ.get("DEEPSEEK_API_KEY") else ("qwen" if os.environ.get("DASHSCOPE_API_KEY") else "not set")
model = os.environ.get("SIMWORLD_LLM_MODEL", "")
if provider == "deepseek":
    auth_ok = bool(os.environ.get("DEEPSEEK_API_KEY"))
    auth_detail = f"DeepSeek / {model or 'deepseek-chat'}" if auth_ok else "NOT SET — run Cell 4"
elif provider == "qwen":
    auth_ok = bool(os.environ.get("DASHSCOPE_API_KEY") or os.environ.get("QWEN_API_KEY"))
    auth_detail = f"Qwen/DashScope / {model or 'qwen-plus'}" if auth_ok else "NOT SET — run Cell 4"
else:
    auth_ok = False
    auth_detail = "NOT SET — run Cell 4"
results["LLM Provider"] = (auth_ok, auth_detail)

# 8. Tunnel
tunnel_url = globals().get('public_url', '')
tunnel_ok = bool(tunnel_url) and tunnel_url.startswith("http")
results["Public URL"] = (tunnel_ok, tunnel_url if tunnel_ok else "Not created")

# Print results
all_ok = True
for name, (ok, detail) in results.items():
    icon = "[PASS]" if ok else "[FAIL]"
    if not ok:
        all_ok = False
    print(f"  {icon} {name}: {detail}")

print()
if all_ok:
    print("ALL CHECKS PASSED! SimWorld Studio is ready.")
    if tunnel_url:
        print(f"Open {tunnel_url} in your browser to start building 3D scenes.")
else:
    print("SOME CHECKS FAILED. Common fixes:")
    print("  - SimWorld MCP: wait 30-60s more, then re-run this cell")
    print("  - LLM Provider: run Cell 4 and enter your DeepSeek or DashScope API key")
    print("  - Tunnel: re-run Cell 6")
    print("  - Studio Backend: !cat /content/studio.log")

## Cell 8: Smoke Test (End-to-End)

Sends a test prompt through the full pipeline: **Chat -> LLM -> MCP Tools -> SimWorld**

In [ ]:
"""End-to-end smoke test — verifies the full pipeline works."""
import requests, json

print("Sending test prompt through the full pipeline...\n")
print("Prompt: 'Set up the environment with a sunny sky, then spawn one small building'\n")

try:
    response = requests.post("http://localhost:3002/api/chat", json={
        "message": "Set up the environment with a sunny sky, then spawn one small residential building (BP_Building_01) at the origin. Then take a screenshot.",
        "sessionId": None,
        "skills": ["building_placement"]
    }, stream=True, timeout=180)

    tool_calls = []
    text_output = []
    screenshots = []
    errors = []
    current_event = None  # Track SSE event type

    for line in response.iter_lines():
        if not line:
            continue
        decoded = line.decode('utf-8')

        # SSE comment lines (heartbeat)
        if decoded.startswith(':'):
            continue

        # Parse SSE event/data pairs
        if decoded.startswith('event: '):
            current_event = decoded[7:]
            continue
        if not decoded.startswith('data: '):
            continue

        try:
            data = json.loads(decoded[6:])
        except json.JSONDecodeError:
            continue

        event_type = current_event or 'unknown'

        if event_type == 'text':
            delta = data.get('delta', '')
            text_output.append(delta)
            print(delta, end='', flush=True)
        elif event_type == 'tool_start':
            name = data.get('displayName', data.get('name', '?'))
            tool_calls.append(name)
            print(f"\n  >> Tool: {name}", flush=True)
        elif event_type == 'tool_result':
            result = data.get('result', '')[:200]
            is_error = data.get('isError', False)
            if is_error:
                errors.append(result)
                print(f"\n  >> ERROR: {result}", flush=True)
            else:
                print(f"\n  >> Result: {result[:100]}...", flush=True)
        elif event_type == 'screenshot':
            screenshots.append(data.get('filepath', ''))
            print(f"\n  >> Screenshot captured!", flush=True)
        elif event_type == 'done':
            cost = data.get('costUsd')
            if cost:
                print(f"\n\n  Cost: ${cost:.4f}")

    print(f"\n\n{'='*50}")
    print(f"Smoke Test Results:")
    print(f"  Tool calls:  {len(tool_calls)} ({', '.join(tool_calls)})")
    print(f"  Screenshots: {len(screenshots)}")
    print(f"  Errors:      {len(errors)}")

    if len(tool_calls) > 0 and len(errors) == 0:
        print(f"\n  SMOKE TEST PASSED!")
        print(f"  Full pipeline working: Chat -> LLM -> MCP -> SimWorld")
    elif len(tool_calls) > 0:
        print(f"\n  PARTIAL PASS — tools fired but some errors occurred.")
    else:
        print(f"\n  SMOKE TEST FAILED — no tool calls detected.")
        print(f"  Check: API key set? SimWorld running? Logs: /content/studio.log")

except requests.exceptions.Timeout:
    print("\nSmoke test timed out (180s). The selected LLM may be slow to respond.")
    print("The platform may still work — try using the browser UI.")
except Exception as e:
    print(f"\nSmoke test error: {e}")
    print("Check /content/studio.log for backend errors.")

---

## Troubleshooting

| Issue | Fix |
|---|---|
| "No GPU detected" | Runtime -> Change runtime type -> GPU (T4) |
| SimWorld TCP fails | Wait 60s more, re-run Cell 7 |
| Studio backend fails | Check: `!cat /content/studio.log` |
| Tunnel fails | Re-run Cell 6, or use Colab proxy |
| LLM errors | Verify your DeepSeek or DashScope API key in Cell 4 |
| Session expires | Colab free tier = 12h max. Re-run all cells. |

### View Logs
```python
!tail -50 /content/ue.log      # SimWorld logs
!tail -50 /content/studio.log    # Studio backend logs
```